In [ ]:
from pathlib import Path

BASE_DIR = Path(".")

CONFIGS = [
    {"cache_dir": "cache_calibration_propaganda950",        "dataset": "English",    "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_propaganda950", "dataset": "English",    "model": "Qwen 8B"},
    {"cache_dir": "cache_calibration_translated",           "dataset": "Translated", "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_translated",    "dataset": "Translated", "model": "Qwen 8B"},
    {"cache_dir": "cache_calibration_ukrainian",            "dataset": "Ukrainian",  "model": "Qwen 4B"},
    {"cache_dir": "cache_calibration_qwen8b_ukrainian",     "dataset": "Ukrainian",  "model": "Qwen 8B"},
]

MODES   = ["text", "image", "image+text"]
N_BINS  = 10
N_FOLDS = 5
SEED    = 42


In [ ]:
import json
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

sys.path.insert(0, str(BASE_DIR.resolve()))
sys.path.insert(0, str(BASE_DIR.parent.resolve()))
from calibration_metrics import (
    multilabel_calibration_report,
    reliability_diagram,
    find_optimal_temperature,
    apply_temperature,
)
from src.hierarchical_f1 import hierarchical_f1


In [ ]:
UNIQUE_LABELS = [
    "Appeal to (Strong) Emotions",
    "Appeal to authority",
    "Appeal to fear/prejudice",
    "Bandwagon",
    "Black-and-white Fallacy/Dictatorship",
    "Causal Oversimplification",
    "Doubt",
    "Exaggeration/Minimisation",
    "Flag-waving",
    "Glittering generalities (Virtue)",
    "Loaded Language",
    "Misrepresentation of Someone's Position (Straw Man)",
    "Name calling/Labeling",
    "Obfuscation, Intentional vagueness, Confusion",
    "Presenting Irrelevant Data (Red Herring)",
    "Reductio ad hitlerum",
    "Repetition",
    "Slogans",
    "Smears",
    "Thought-terminating cliché",
    "Transfer",
    "Whataboutism",
    "NO_PROPAGANDA",
]


In [ ]:
def _load_json(p):
    text = p.read_text()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        if "Extra data" in str(e):
            return json.loads(text[:e.pos])
        raise


def load_records(cache_dir, mode):
    mode_subdir = mode.replace("+", "_")
    manifest = json.loads((BASE_DIR / cache_dir / mode_subdir / "manifest.json").read_text())
    records = []
    for entry in manifest:
        p = Path(entry["path"])
        if not p.is_absolute():
            p = BASE_DIR / p
        records.append(_load_json(p))
    return records


def clean_labels(labels):
    s = set(labels) if labels else set()
    return {"NO_PROPAGANDA"} if "NO_PROPAGANDA" in s else s


def build_arrays(records):
    per_conf = {lbl: [] for lbl in UNIQUE_LABELS}
    per_corr = {lbl: [] for lbl in UNIQUE_LABELS}
    for r in records:
        gold = clean_labels(r.get("gold_labels", []))
        conf_dict = r.get("confidence") or {}
        for label in UNIQUE_LABELS:
            conf_val = conf_dict.get(label, 0.0)
            try:
                conf_val = float(conf_val)
                if not (0.0 <= conf_val <= 1.0):
                    conf_val = 0.0
            except (TypeError, ValueError):
                conf_val = 0.0
            per_conf[label].append(conf_val)
            per_corr[label].append(1.0 if label in gold else 0.0)
    return {lbl: (np.array(per_conf[lbl]), np.array(per_corr[lbl])) for lbl in UNIQUE_LABELS}


In [ ]:
EPS = 1e-10


def safe_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


@dataclass
class PlattScaler:
    A: float
    B: float
    n_train: int

    def predict(self, conf):
        return 1.0 / (1.0 + np.exp(-(self.A * safe_logit(conf) + self.B)))


def fit_platt(conf, corr):
    X = safe_logit(conf).reshape(-1, 1)
    y = corr.astype(int)
    lr = LogisticRegression(C=1e10, solver="lbfgs", max_iter=1000)
    lr.fit(X, y)
    return PlattScaler(A=float(lr.coef_[0, 0]), B=float(lr.intercept_[0]), n_train=len(conf))


def platt_oof(conf, corr, n_folds=N_FOLDS, seed=SEED):
    oof_cal = conf.copy()
    scalers = []
    y = corr.astype(int)

    if len(np.unique(y)) < 2:
        return oof_cal, []

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for train_idx, val_idx in skf.split(conf, y):
        if len(np.unique(y[train_idx])) < 2:
            continue
        scaler = fit_platt(conf[train_idx], corr[train_idx])
        oof_cal[val_idx] = scaler.predict(conf[val_idx])
        scalers.append(scaler)

    return oof_cal, scalers


In [ ]:
all_data  = {}
all_platt = {}
all_temp  = {}

for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        arrays = build_arrays(load_records(cfg["cache_dir"], mode))

        class_conf = {lbl: arrays[lbl][0] for lbl in UNIQUE_LABELS}
        class_corr = {lbl: arrays[lbl][1] for lbl in UNIQUE_LABELS}
        all_data[key] = {"class_conf": class_conf, "class_corr": class_corr}

        pooled_conf = np.concatenate(list(class_conf.values()))
        pooled_corr = np.concatenate(list(class_corr.values()))
        T = find_optimal_temperature(pooled_conf, pooled_corr)
        all_temp[key] = {
            "class_conf": {lbl: apply_temperature(class_conf[lbl], T) for lbl in UNIQUE_LABELS},
            "class_corr": class_corr,
            "T": T,
        }

        platt_conf = {}
        for lbl in UNIQUE_LABELS:
            oof, _ = platt_oof(class_conf[lbl], class_corr[lbl])
            platt_conf[lbl] = oof
        all_platt[key] = {"class_conf": platt_conf, "class_corr": class_corr}


In [ ]:
reports = {}
for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        for method, store in [("raw", all_data), ("temperature", all_temp), ("platt", all_platt)]:
            reports[(cfg["dataset"], cfg["model"], mode, method)] = multilabel_calibration_report(
                store[key]["class_conf"], store[key]["class_corr"], n_bins=N_BINS,
            )


In [ ]:
rows = []
for cfg in CONFIGS:
    for mode in MODES:
        for method in ("raw", "temperature", "platt"):
            rep = reports[(cfg["dataset"], cfg["model"], mode, method)]
            rows.append({
                "dataset":     cfg["dataset"],
                "model":       cfg["model"],
                "mode":        mode,
                "method":      method,
                "macro_ece":   rep.macro_ece,
                "micro_ece":   rep.micro_ece,
                "macro_brier": rep.macro_brier,
            })

df = pd.DataFrame(rows)

pivot = df.pivot_table(index=["dataset", "model", "mode"], columns="method", values="macro_ece").round(4)
pivot = pivot[["raw", "temperature", "platt"]]
pivot["delta_platt_raw"] = (pivot["platt"] - pivot["raw"]).round(4)
pivot.style.background_gradient(subset=["raw", "temperature", "platt"], cmap="YlOrRd", axis=None) \
           .background_gradient(subset=["delta_platt_raw"], cmap="RdYlGn_r", axis=None) \
           .format("{:.4f}")


In [ ]:
DEMO_DATASET = "Translated"
DEMO_MODEL   = "Qwen 4B"
DEMO_MODE    = "text"

key = (DEMO_DATASET, DEMO_MODEL, DEMO_MODE)
methods = [
    ("raw",         all_data[key]["class_conf"]),
    ("temperature", all_temp[key]["class_conf"]),
    ("platt",       all_platt[key]["class_conf"]),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label, class_conf) in zip(axes, methods):
    all_conf = np.concatenate(list(class_conf.values()))
    all_corr = np.concatenate(list(all_data[key]["class_corr"].values()))
    rep = reports[(DEMO_DATASET, DEMO_MODEL, DEMO_MODE, label)]
    reliability_diagram(all_conf, all_corr, n_bins=N_BINS, title=f"{label} (micro ECE={rep.micro_ece:.4f})", ax=ax)

plt.suptitle(f"Reliability — {DEMO_DATASET} | {DEMO_MODEL} | {DEMO_MODE}")
plt.tight_layout()
plt.show()


## Cross-Dataset Transfer

In [ ]:
full_scalers = {}
for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        cc = all_data[key]["class_conf"]
        cr = all_data[key]["class_corr"]
        label_scalers = {}
        for lbl in UNIQUE_LABELS:
            y = cr[lbl].astype(int)
            label_scalers[lbl] = fit_platt(cc[lbl], cr[lbl]) if len(np.unique(y)) >= 2 else None
        full_scalers[key] = label_scalers


In [ ]:
TRANSFER_MODE = "text"
DATASETS = ["English", "Translated", "Ukrainian"]

transfer_rows = []
for model in ["Qwen 4B", "Qwen 8B"]:
    for src_dataset in DATASETS:
        src_scalers = full_scalers[(src_dataset, model, TRANSFER_MODE)]
        for tgt_dataset in DATASETS:
            tgt_key = (tgt_dataset, model, TRANSFER_MODE)
            tgt_conf = all_data[tgt_key]["class_conf"]
            tgt_corr = all_data[tgt_key]["class_corr"]

            transferred = {}
            for lbl in UNIQUE_LABELS:
                scaler = src_scalers.get(lbl)
                transferred[lbl] = scaler.predict(tgt_conf[lbl]) if scaler is not None else tgt_conf[lbl]

            rep_raw = reports[(tgt_dataset, model, TRANSFER_MODE, "raw")]
            rep_xfer = multilabel_calibration_report(transferred, tgt_corr, n_bins=N_BINS)

            transfer_rows.append({
                "model":              model,
                "source":             src_dataset,
                "target":             tgt_dataset,
                "raw_macro_ece":      rep_raw.macro_ece,
                "transfer_macro_ece": rep_xfer.macro_ece,
                "delta":              round(rep_xfer.macro_ece - rep_raw.macro_ece, 4),
                "same_domain":        src_dataset == tgt_dataset,
            })

transfer_df = pd.DataFrame(transfer_rows)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model in zip(axes, ["Qwen 4B", "Qwen 8B"]):
    sub = transfer_df[transfer_df["model"] == model]
    mat = sub.pivot(index="target", columns="source", values="delta").loc[DATASETS, DATASETS]
    sns.heatmap(mat, annot=True, fmt=".4f", cmap="RdYlGn_r", center=0, linewidths=0.5, ax=ax,
                cbar_kws={"label": "Δ macro ECE (transfer − raw)"})
    ax.set_title(f"{model} — Δ ECE applying source scaler to target")
    ax.set_xlabel("Source dataset")
    ax.set_ylabel("Target dataset")

plt.suptitle(f"Cross-Dataset Platt Transfer — mode={TRANSFER_MODE}")
plt.tight_layout()
plt.show()


## Hierarchical F1

In [ ]:
def gold_for_eval(labels):
    s = set(labels) if labels else set()
    return set() if "NO_PROPAGANDA" in s else s


hf1_results = {}
for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        records = load_records(cfg["cache_dir"], mode)
        gold = [gold_for_eval(r.get("gold_labels", [])) for r in records]

        pred_raw = [set(r.get("pred_labels", [])) for r in records]
        platt_conf = all_platt[key]["class_conf"]
        pred_cal = [
            {lbl for lbl in UNIQUE_LABELS if platt_conf[lbl][i] > 0.5}
            for i in range(len(records))
        ]

        hf1_results[key] = {
            "raw":        hierarchical_f1(gold, pred_raw),
            "calibrated": hierarchical_f1(gold, pred_cal),
        }

hf1_rows = []
for key, res in hf1_results.items():
    dataset, model, mode = key
    for method, r in res.items():
        hf1_rows.append({
            "dataset":   dataset,
            "model":     model,
            "mode":      mode,
            "method":    method,
            "micro_f1":  r["micro"]["f1"],
            "macro_f1":  r["macro_per_label"]["f1"],
            "sample_f1": r["per_sample"]["f1"],
        })

pd.DataFrame(hf1_rows).round(4)


In [ ]:
from sklearn.metrics import roc_auc_score

auc_rows = []
for cfg in CONFIGS:
    for mode in MODES:
        key = (cfg["dataset"], cfg["model"], mode)
        cc = all_data[key]["class_conf"]
        cr = all_data[key]["class_corr"]

        aucs = []
        for lbl in UNIQUE_LABELS:
            y = cr[lbl].astype(int)
            if len(np.unique(y)) < 2:
                continue
            aucs.append(roc_auc_score(y, cc[lbl]))

        auc_rows.append({
            "dataset":  cfg["dataset"],
            "model":    cfg["model"],
            "mode":     mode,
            "mean_auc": float(np.mean(aucs)),
            "min_auc":  float(np.min(aucs)),
            "n_labels": len(aucs),
        })

pd.DataFrame(auc_rows).pivot_table(index=["dataset", "model"], columns="mode", values="mean_auc").round(4)


In [ ]:
out = {"calibration": {}, "transfer": [], "prediction_hf1": {}}

for cfg in CONFIGS:
    for mode in MODES:
        cfg_key = f"{cfg['dataset']}_{cfg['model'].replace(' ', '_')}_{mode}"
        out["calibration"][cfg_key] = {}
        for method in ("raw", "temperature", "platt"):
            rep = reports[(cfg["dataset"], cfg["model"], mode, method)]
            out["calibration"][cfg_key][method] = {
                "macro_ece":   rep.macro_ece,
                "micro_ece":   rep.micro_ece,
                "macro_ace":   rep.macro_ace,
                "macro_brier": rep.macro_brier,
                "micro_brier": rep.micro_brier,
            }
        out["calibration"][cfg_key]["temperature_T"] = all_temp[(cfg["dataset"], cfg["model"], mode)]["T"]

        key = (cfg["dataset"], cfg["model"], mode)
        if key in hf1_results:
            out["prediction_hf1"][cfg_key] = {
                method: {"micro": r["micro"], "macro_per_label": r["macro_per_label"], "per_sample": r["per_sample"]}
                for method, r in hf1_results[key].items()
            }

out["transfer"] = transfer_df.to_dict(orient="records")

(BASE_DIR / "platt_scaling_results.json").write_text(json.dumps(out, indent=2, default=str))
